## Unload Ragu

Queries all specified sandbox tables and saves each as Parquet to S3 using ODBC connection

In [2]:
import os
import pandas as pd
import pyodbc
import warnings

#### Tables to Unload

In [3]:
TABLES = (

"sandbox.gl_ragu_individual_ste",
# "sandbox.gl_ragu_individual_nofraud",
    # "sandbox.temp_employment_type_ragu",
    # "sandbox.temp_prov_customer_credit_attributes_ragu",
    # "sandbox.temp_los_customer_credit_attributes_ragu",
    # "sandbox.temp_fraud_ragu",
    # "sandbox.temp_blackbook_values_ragu",
    # "sandbox.rds_blackbook_rollup",
    # "sandbox.rds_blackbook_rollup_temp",
    # "sandbox.rds_rec_model_originations",
    # "sandbox.rds_rec_model_originations_temp",

)

OUTPUT_DIR = r"S:\rds\raguUnload"

print(f"Tables to unload: {len(TABLES)}")
for t in TABLES:
    print(f"  {t}")

Tables to unload: 1
  sandbox.gl_ragu_individual_ste


#### Run Unload

In [4]:
results = []

with pyodbc.connect("DSN=Redshift_prod_new") as conn:
    for table in TABLES:
        table_name = table.split(".")[-1]
        output_path = os.path.join(OUTPUT_DIR, f"{table_name}.parquet")
        print(f"Querying {table}...", end=" ")

        try:
            warnings.filterwarnings("ignore", category=UserWarning)
            df = pd.read_sql_query(sql=f"SELECT * FROM {table}", con=conn)
            warnings.filterwarnings("default", category=UserWarning)

            df.to_parquet(output_path, index=False)
            print(f"{len(df)} rows -> {output_path}")
            results.append({"table": table, "rows": len(df), "cols": len(df.columns), "status": "OK"})
        except Exception as e:
            print(f"FAILED: {e}")
            results.append({"table": table, "rows": 0, "cols": 0, "status": str(e)})

print("\nDone.")

Querying sandbox.gl_ragu_individual_ste... 20042 rows -> S:\rds\raguUnload\gl_ragu_individual_ste.parquet

Done.


#### Summary

In [5]:
pd.DataFrame(results)


,table,rows,cols,status
0,sandbox.gl_ragu_individual_ste,20042,11,OK
